# Member 3 — Outlier Removal

**Technique:** Remove exact duplicate images and resolution/brightness outliers using the IQR rule.

## Why this dataset needs it
Duplicate leaves leak into both train and test if not removed. Extreme tiny/huge or near-black frames are often capture artefacts and distort color histograms.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Resolve Group_Deliverable root whether cwd is notebooks/ or deliverable root
HERE = Path.cwd().resolve()
ROOT = None
for p in (HERE, *HERE.parents):
    if (p / "src" / "preprocess_utils.py").exists():
        ROOT = p
        break
    if (p / "Group_Deliverable" / "src" / "preprocess_utils.py").exists():
        ROOT = p / "Group_Deliverable"
        break
if ROOT is None:
    raise FileNotFoundError("Run from progress/Group_Deliverable or its notebooks/ folder.")

sys.path.insert(0, str(ROOT / "src"))
from preprocess_utils import (
    SEED, SOURCE_URL, DOI, FOLDERS, CLASSES, paths,
    inventory_table, discover_images, audit_images,
    remove_exact_duplicates, iqr_mask, extract_feature_matrix, read_rgb,
    stratified_sample,
)

P = paths(ROOT)
RAW, VIZ, OUT, LOGS = P["raw"], P["viz"], P["outputs"], P["logs"]
for d in (VIZ, OUT, LOGS):
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(SEED)
print("Deliverable root:", ROOT)
print("Raw data:", RAW)
print("Dataset:", SOURCE_URL, "| DOI:", DOI)


## 1. Start from encoded metadata / audit, drop duplicates & IQR outliers


In [ ]:
src = OUT / "m2_encoded_metadata.csv"
if src.exists():
    df = pd.read_csv(src)
else:
    df, _ = audit_images(RAW)
    print("Warning: m2 output missing; using fresh audit.")

before = len(df)
df_unique, n_dup = remove_exact_duplicates(df)
print(f"Exact pixel duplicates removed: {n_dup} (kept {len(df_unique)} / {before})")

# IQR on resolution and green-channel brightness (basil is green-dominant)
mask_px = iqr_mask(df_unique["pixels"], k=1.5)
mask_g = iqr_mask(df_unique["mean_g"], k=1.5)
keep = mask_px & mask_g
removed = df_unique.loc[~keep].copy()
clean = df_unique.loc[keep].reset_index(drop=True)

print(f"IQR outliers removed: {len(removed)}")
print(f"Final cleaned rows: {len(clean)}")
print(clean["label"].value_counts())

clean.to_csv(OUT / "m3_cleaned_no_outliers.csv", index=False)
removed.to_csv(OUT / "m3_removed_outliers.csv", index=False)
(LOGS / "m3_outlier_summary.json").write_text(
    json.dumps({
        "before": before,
        "duplicates_removed": int(n_dup),
        "iqr_removed": int(len(removed)),
        "after": int(len(clean)),
        "class_counts": {str(k): int(v) for k, v in clean["label"].value_counts().items()},
    }, indent=2),
    encoding="utf-8",
)


## 2. EDA visualization — boxplots before filtering


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df_unique, x="label", y="pixels", hue="label", order=CLASSES, ax=axes[0], palette=["#2ca02c", "#d62728"], legend=False)
axes[0].set_title("Image resolution (pixels) by class")
axes[0].set_ylabel("width × height")

sns.boxplot(data=df_unique, x="label", y="mean_g", hue="label", order=CLASSES, ax=axes[1], palette=["#2ca02c", "#d62728"], legend=False)
axes[1].set_title("Mean green intensity by class")
axes[1].set_ylabel("Mean G (0–255)")

fig.tight_layout()
fig.savefig(VIZ / "m3_outlier_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Interpretation: points far outside the whiskers are candidates for IQR removal; both classes should still remain after cleaning.")


## Viva talking points
1. Explain duplicate leakage risk.
2. Explain IQR fences on `pixels` and `mean_g`.
3. Interpret the boxplots and confirm both classes remain.
